#### Imports

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))


import torch
import torch.nn as nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

import pytorch_lightning as pl

import wandb
wandb.login()


from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import RichProgressBar, ModelCheckpoint
from pytorch_lightning.tuner import Tuner

from utils import seed_everything, VolumeDataModule2D
from models_2d import UNet, Pl_Model

cuda


wandb: Currently logged in as: nbennewiz to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
seed_everything(100)

Seed set to 100


100

#### Config

In [3]:
config = {
    "root": "../NormalizedQualityFiltered",
    'batch_size': 2,
    'learning_rate': 0.0001,
    "num_workers": 10,#0, wenn die gpu nicht benutzt wird
    "pin_memory": True if torch.cuda.is_available() else False,#False, wenn die gpu nicht benutzt wird
    "drop_last": False,
    'epochs': 40,
    #'log_interval': 20,
    #'viz_interval': 1,
    'run_name': '2D-UNet',
    'input_frames': 9,
    "pred_frames": 9,
    "pred_n_frames_per_step": 9,
    "train_split": 0.7,
    "val_split": 0.15,
    "test_split": 0.15,
}

config["run_name"] += f"_{config['pred_frames']}"
if config["pred_frames"] == config["pred_n_frames_per_step"]:
    config["run_name"] += "_NAR"
elif config["pred_n_frames_per_step"] == 1:
    config["run_name"] += "_FAR"
else:
    config["run_name"] += f"_PAR_{config['pred_n_frames_per_step']}"

# Get data module
dm = VolumeDataModule2D(
    root=config["root"],
    batch_size=config['batch_size'],
    num_workers=config["num_workers"],
    pin_memory=config["pin_memory"],
    drop_last=config["drop_last"],
    sequence_length=config["input_frames"],
    prediction_length=config["pred_frames"],
    train_split=config["train_split"],
    val_split=config["val_split"],
    test_split=config["test_split"],
)

wandb_logger = WandbLogger(entity="ChadCTP", project="perfusion-ct-prediction", name=config["run_name"])

# Initialize model for tuning
model = UNet()

# Initialize pl_model for tuning
pl_model = Pl_Model(
    passed_model=model,
    config=config,
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_total_loss/dataloader_idx_0",  
    mode="min",  
    save_top_k=1,  
    filename="best-checkpoint",
    verbose=True,
)

# Initialize trainer for tuning
trainer = pl.Trainer(
    logger=wandb_logger,
    accelerator="gpu",
    devices= [1] if torch.cuda.is_available() else None,
    max_epochs=config["epochs"],
    callbacks=[RichProgressBar(), checkpoint_callback],
    check_val_every_n_epoch=5,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [4]:
trainer.fit(
    model=pl_model,
    datamodule=dm,
)

torch.Size([2, 9, 1, 256, 256])

RuntimeError: Given groups=1, weight of size [32, 1, 3, 3, 3], expected input[2, 9, 1, 256, 256] to have 1 channels, but got 9 channels instead

In [9]:
#check and log the losses "to beat"
dm.setup()
pl_model.check_losses(dm.train_dataloader(), mode="train", use_wandb=True)
pl_model.check_losses(dm.val_dataloader()[0], mode="val", use_wandb=True)
pl_model.check_losses(dm.test_dataloader(), mode="test", use_wandb=True)

#val_results = trainer.validate(pl_model, datamodule=dm)
test_results = trainer.test(pl_model, datamodule=dm)

save_load_path = f"../ModelWeights/{config['run_name']}.ckpt"
trainer.save_checkpoint(save_load_path)

Testing ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 0:00:07 • 0:00:00 2.27it/s

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      test_huber_loss      │   0.0036088123451918364   │
│    test_huberssim_loss    │    0.12976357340812683    │
│       test_mse_loss       │    0.01101748924702406    │
│      test_psnr_loss       │    19.582061767578125     │
│      test_rmse_loss       │    0.1049467995762825     │
│      test_ssim_loss       │    0.4232603907585144     │
│    test_temporal_loss     │   0.0025927398819476366   │
│      test_total_loss      │    0.12976357340812683    │
└───────────────────────────┴───────────────────────────┘

#testing
pl_model = Pl_Model.load_from_checkpoint(
    "./perfusion-ct-prediction/oqelujc5/checkpoints/best-checkpoint.ckpt",
    passed_model=model,
)

dm.setup()
test_dataloader = dm.test_dataloader()

print(len(test_dataloader))
pl_model.to(device)
"""for inputs, targets in test_dataloader:
    outputs = pl_model.forward(inputs.to(device))
    outputs = outputs.detach().cpu()
    outputs = torch.concat([inputs, outputs], dim=1).squeeze(0, 2).numpy()
    targets = torch.concat([inputs, targets], dim=1).squeeze(0, 2).numpy()
    print(outputs.shape)
    print(targets.shape)
    break"""
# method 1
#inputs, targets = next(iter(test_dataloader))

# method 2
# only for single shot
file_name = "MOL-253"
vol = torch.tensor(np.load(f"../NormalizedQualityFiltered/{file_name}.npy")).unsqueeze(1)
inputs = vol[0:9]#.unsqueeze(0)
targets = vol[9:]#.unsqueeze(0)
print(inputs.shape)
inputs_ = torch.concat([inputs.transpose(0, 2).transpose(1, 2), torch.zeros([config["batch_size"]-vol.shape[2], config["input_frames"], 1, vol.shape[-2], vol.shape[-1]])])
outputs = pl_model(inputs_.to(device))
outputs = outputs[:vol.shape[2]]
print(outputs.shape)
outputs = outputs.detach().cpu()
outputs = torch.concat([inputs, outputs.transpose(0, 2).transpose(0, 1)], dim=0).squeeze(1).numpy()
targets = torch.concat([inputs, targets], dim=1).squeeze(1).numpy()

print(outputs.shape)
print(targets.shape)
#for i in range(inputs.shape[2]):
#    print(inputs[:, :, i, :, :].shape)

np.save(f"outputs_{file_name}_{config["run_name"]}.npy", outputs)
np.save(f"targets_{file_name}_{config["run_name"]}.npy", targets)

@torch.no_grad()
def overall_loss(model, loader, device):
    mse_loss = 0.0
    huber_loss = 0.0
    rmse_loss = 0.0
    #ssim_loss = 0.0
    psnr_loss = 0.0
    total_loss = 0.0
    model = model.to(device)
    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        # iterate over depth
        outputs = []
        for d in range(inputs.shape[3]):
            inputs_ = inputs[:, :, :, d, :, :]
            #iterate over time
            outputs_d = []
            for t in range(0, model.config["pred_frames"], model.config["pred_n_frames_per_step"]):
                if model.config["pred_frames"]-t<model.config["pred_n_frames_per_step"]:
                    frames_this_step = model.config["pred_frames"]-t
                else:
                    frames_this_step = model.config["pred_n_frames_per_step"]
                
                outputs_t = model.forward(inputs_)
        
                #get only the first predicted frame
                outputs_t = outputs_t[:, :frames_this_step, :, :]

                #add to depth lst
                outputs_d.append(outputs_t)
        
                inputs_ = torch.cat([inputs_[:, model.config["pred_n_frames_per_step"]:, :, :], outputs_t], dim=1)
            #concat time and add to overall lst
            outputs_d = torch.concat(outputs_d, dim=1)
            outputs.append(outputs_d)
    
        #stack over depth
        outputs = torch.stack(outputs, dim=3)
        #print(outputs.shape)
        
        #calculate losses
        mse_loss += model.mse_criterion(outputs, targets).item()
        huber_loss += model.huber_criterion(outputs, targets).item()
        rmse_loss += torch.sqrt(model.mse_criterion(outputs, targets)).item()
        #ssim_loss = model.ssim_criterion(outputs, targets).item()
        psnr_loss += model.psnr_criterion(outputs, targets).item()
        total_loss += huber_loss

    mse_loss = mse_loss / len(loader)
    huber_loss = huber_loss / len(loader)
    rmse_loss = rmse_loss / len(loader)
    #ssim_loss = ssim_loss / len(loader)
    psnr_loss = psnr_loss / len(loader)
    total_loss = total_loss / len(loader)

    return outputs, mse_loss, huber_loss, rmse_loss, psnr_loss, total_loss

dm.setup()
_, mse_loss, huber_loss, rmse_loss, psnr_loss, total_loss = overall_loss(model=pl_model, loader=dm.test_dataloader_3d(), device=device)
mse_loss, huber_loss, rmse_loss, psnr_loss, total_loss

'''# idk tuning behaves really weird and different from fit regarding memory usage
#tuning
tuner = Tuner(trainer)
tuner.scale_batch_size(pl_model, datamodule=dm, mode="binsearch")

#cleaning up
del tuner
del trainer
del pl_model
del model
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    torch.cuda.ipc_collect()

# Reinitialize model for tuning
model = SimVP(
    shape_in=[config["input_frames"], 1, 256, 256]
)

# Reinitialize pl_model for training
pl_model = Pl_Model(
    passed_model=model,
    config=config,
)

# Reinitialize trainer for training
trainer = pl.Trainer(
    logger=wandb_logger,
    accelerator="gpu",
    devices= [2] if torch.cuda.is_available() else None,
    max_epochs=config["epochs"],
    callbacks=[RichProgressBar()],
    check_val_every_n_epoch=1,
    #enable_checkpointing=False,
)'''